# F006.3 — Multilingual academic + modern generator hardening (Track A: AI writing detection)

Runs on a fresh Colab VM and resumes after a disconnect (re-run cells 1-14, then the launch cell; finished steps leave `.done` markers and
training resumes from the Drive `last` checkpoint). Drive `MyDrive/turniting/f006_3/` keeps every dataset, manifest, checkpoint, model,
report and log. F006.1, F006.1.1, F006.2, F008*, V2, V3 are read-only. Everything stays EXPERIMENTAL; ACTIVE_MODEL = NONE.

Order: (A) `f0063_gen_stage.sh` = OpenAlex (es/en/pt/fr/de + it/nl) + AIGTBench + open generators (Qwen2.5-14B, Mistral-Nemo-12B TRAIN;
OLMo-2-7B unseen proxy; Phi-3.5-mini, Granite-3.1-8B FINAL only). (B) `f0063_pipeline.sh` = Claude data (aux, never FINAL) -> dedup vs all
history -> language metadata -> group-safe splits + mixed-language documents -> leakage gates (STOP_BEFORE_TRAINING if != 0) -> LOCKED
F006_3_FINAL_EXTERNAL_HOLDOUT -> B1-B5 from F006.2 Candidate B -> compare vs F006.1.1 / F006.2 A / F006.2 B with the pre-registered gates
-> MODEL_LOCK -> FINAL once -> report.

In [ ]:
# 1. Mount Google Drive (needs your click)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Paths (new independent area)
import os, json, shutil, subprocess, hashlib
T = '/content/drive/MyDrive/turniting'
D = f'{T}/f006_3'
for s in ['datasets/raw', 'datasets/normalized', 'datasets/multilingual', 'datasets/mixed_language', 'datasets/splits', 'datasets/manifests',
          'models/checkpoints', 'models/final', 'evaluation', 'reports', 'logs']:
    os.makedirs(f'{D}/{s}', exist_ok=True)
REPO, BRANCH = '/content/Turniting', 'exp/f006-3-multilingual-academic-hardening'

In [ ]:
# 3. Clone or update the repository (code only; Colab never pushes)
if not os.path.exists(f'{REPO}/.git'):
    !git clone -q --branch {BRANCH} https://github.com/Anthgg/Turniting.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
%cd /content/Turniting/training
!git log --oneline -1

In [ ]:
# 4. Dependencies (the Colab image's torch is kept)
!pip -q install transformers==4.46.3 accelerate==1.1.1 sentencepiece==0.2.0 datasketch==1.6.5 protobuf langid
import transformers; print('transformers', transformers.__version__)

In [ ]:
# 5-9. GPU: name, VRAM, CUDA, bf16 / fp16 (the training scripts autodetect the same way and probe the batch size)
import torch
assert torch.cuda.is_available(), 'switch the runtime to a GPU'
p = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported()
print('GPU =', p.name); print('VRAM =', f'{p.total_memory/2**30:.1f} GiB'); print('CUDA =', torch.version.cuda)
print('PRECISION =', 'bf16' if BF16 else 'fp16 (+ loss scaling)')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 10. Historical inputs (read-only)
need = {'F006.1.1 model (primary baseline + KD teacher)': f'{T}/f006_1_1/models/experimental_f006_1_1/model_manifest.json',
        'F006.2 Candidate A': f'{T}/f006_2/models/final/experimental_f006_2/model_manifest.json',
        'F006.2 Candidate B (init of B1-B5)': f'{T}/f006_2/models/checkpoints/runs/F0062_B/best',
        'F006.2 train_extra (replay incl. F006.1.1 + TDX controlled replay)': f'{T}/f006_2/datasets/splits/f006_2_train_extra.parquet',
        'F006.2 final (REGRESSION_HOLDOUT)': f'{T}/f006_2/datasets/splits/f006_2_final_external_holdout.parquet',
        'F006.1.1 final (REGRESSION_HOLDOUT)': f'{T}/f006_1_1/datasets/f006_1_1_final_holdout.parquet',
        'F006.1.1 full run config': f'{T}/f006_1_1/runs/F00611_FULL/run.json', 'V2 master': f'{T}/v3/datasets/master_v2'}
for k, v in need.items():
    print('OK ' if os.path.exists(v) else 'MISSING', k)
assert all(os.path.exists(v) for v in need.values())

In [ ]:
# 11-14. Manifests, hashes, split integrity and leakage (re-run any time)

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(8 << 20), b''): h.update(c)
    return h.hexdigest()

M, S = f'{D}/datasets/manifests', f'{D}/datasets/splits'
if os.path.exists(f'{M}/f006_3_sources.json'):
    print(json.dumps(json.load(open(f'{M}/f006_3_sources.json')), indent=1)[:1500])
for g in ['qwen2.5-14b', 'mistral-nemo-12b', 'olmo2-7b', 'phi3.5-mini', 'granite3.1-8b']:
    print('gen', g, 'done' if os.path.exists(f'{D}/logs/gen_{g}.done') else 'pending')
hm = f'{M}/f006_3_final_external_holdout.json'
if os.path.exists(hm):
    h = json.load(open(hm))
    print('FINAL LOCKED:', h['LOCKED'], '| sha ok:', sha(f'{S}/f006_3_final_external_holdout.parquet') == h['holdout_file_sha256'])
    print('leakage:', h['leakage']['result'], h['gates'], '| Claude rows in FINAL:', h['leakage']['FINAL_claude_rows'])
else:
    print('splits not frozen yet')

In [ ]:
# Launch f0063_gen_stage.sh detached (survives cell interrupts; never use `!cmd &` and never interrupt the cell that launched a job)
pid = '/content/f0063_gen.pid'
if os.path.exists(pid) and os.path.exists(f"/proc/{open(pid).read().strip()}"):
    print('already running')
else:
    out = open('/content/f0063_gen.out', 'a')
    pr = subprocess.Popen(['bash', 'scripts/f0063_gen_stage.sh'], stdout=out, stderr=subprocess.STDOUT, start_new_session=True)
    open(pid, 'w').write(str(pr.pid)); print('started pid', pr.pid)

In [ ]:
# Launch f0063_pipeline.sh detached (survives cell interrupts; never use `!cmd &` and never interrupt the cell that launched a job)
pid = '/content/f0063_pipeline.pid'
if os.path.exists(pid) and os.path.exists(f"/proc/{open(pid).read().strip()}"):
    print('already running')
else:
    out = open('/content/f0063_pipeline.out', 'a')
    pr = subprocess.Popen(['bash', 'scripts/f0063_pipeline.sh'], stdout=out, stderr=subprocess.STDOUT, start_new_session=True)
    open(pid, 'w').write(str(pr.pid)); print('started pid', pr.pid)

In [ ]:
# Monitor (re-run any time)
!tail -n 25 /content/f0063_gen.out 2>/dev/null; echo ----; tail -n 30 /content/f0063_pipeline.out 2>/dev/null; echo; ls {D}/logs
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

In [ ]:
# Report bundle for Git (download and hand it to Claude; Colab cannot push)
!python -W ignore scripts/f0063_report.py
!tar czf {D}/f006_3_reports_bundle.tgz -C /content/Turniting/training reports/f006_3 F006_3_FINAL_REPORT.md F006_3_FINAL_REPORT.json datasets/manifests/model_registry_f006_3.json 2>/dev/null; ls -la {D}/f006_3_reports_bundle.tgz
from google.colab import files
files.download(f'{D}/f006_3_reports_bundle.tgz')